In [182]:
gen_report = False

In [183]:
import os
import json
import pandas as pd

root = "experiments"

records = []

for dirpath, dirnames, filenames in os.walk(root):
    if "report.json" in filenames:
        report_path = os.path.join(dirpath, "report.json")

        with open(report_path, "r") as f:
            data = json.load(f)

        record = {}

        record["VQEL"] = data.get("VQEL")

        # metrics
        for k, v in data.get("metrics", {}).items():
            record[k] = v

        # config
        for k, v in data.get("config", {}).items():
            if 'dir' in k.lower() or 'split' in k.lower():
                continue
            record[k] = v

        record["path"] = dirpath.replace('experiments/', '').replace('/results', '')
        records.append(record)

df = pd.DataFrame(records)

df["reset_unfrozen_params"] = df["reset_unfrozen_params"].map({True: "yes", False: "no"})
df["freeze_codebook"] = df["freeze_codebook"].map({True: "yes", False: "no"})
df["freeze_object_encoder"] = df["freeze_object_encoder"].map({True: "yes", False: "no"})

df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
df.loc[df["agent_a_training_mode"] == 'frozen', "learning_rate_phase2_a"] = "-"

df = df.where(pd.notna(df), "None")
pd.options.display.float_format = '{:.10g}'.format
df = df.map(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and (abs(x) < 0.01 or abs(x) == 0.1 or abs(x) == 0.01) else x)
print("Total reports loaded:", len(df))


Total reports loaded: 28


/tmp/ipykernel_557231/1038596731.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
/tmp/ipykernel_557231/1038596731.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["agent_a_training_mode"] == 'frozen', "learning_rate_phase2_a"] = "-"


In [184]:
def filter_df(filters, df=df):

    idx = pd.Series(True, index=df.index)

    for key, values in filters.items():
        mask = pd.Series(False, index=df.index)
        if not isinstance(values, (list, tuple, set)):
            values = [values]
        for value in values:
            if value == "!None":
                mask |= (df[key] != "None")
            else:
                mask |= (df[key] == value)
        idx &= mask 

    return df[idx].sort_values(by=["message_length", "agent_a_training_mode", "learning_rate_phase1", "learning_rate_phase2_b", "learning_rate_phase2_a", "learning_rate_tt", "num_iterations"])
    

In [185]:
import base64

def to_html(df):    

    # ---- highlight rule ----
    if 'mutual_play_accuracy' in df:
        max_col = 'mutual_play_accuracy'
    else:
        max_col = 'test_accuracy'
    max_val = pd.to_numeric(df[max_col]).max()

    def highlight_max_row(row):
        if  pd.to_numeric(row[max_col]) == max_val:
            return ['font-weight: bold; background-color: #ffff99'] * len(row)
        else:
            return [''] * len(row)

    # ---- style ----
    styled = (
        df.style
            .apply(highlight_max_row, axis=1)  # <<< APPLY HIGHLIGHT HERE
            .hide(axis="index")
            .format(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and abs(x) < 0.01 else x)
            .set_table_styles([
                {"selector": "td", "props": [("border-right", "1px solid black")]},
                {"selector": "th", "props": [
                    ("border-right", "1px solid black"),
                    ("color", "darkblue"),
                    ("font-weight", "bold"),
                    ("padding-left", "8px"),
                    ("padding-right", "8px")
                ]},
            ])
            .set_properties(**{"text-align": "center"})
    )

    html_table = styled.to_html(index=False)

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_table)

        
def add_heading(num=1, title=""):
    """Add a professional-colored heading to results.html"""
    colors = ["#0b3d91",  # dark blue
              "#800000",  # maroon
              "#205522",  # dark green
              "#4b0082",  # indigo
              "#444444"]  # dark gray
    color = colors[(num-1) % len(colors)]  # cycle through colors
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f'<h{num} style="color:{color}; font-family:Arial, sans-serif;">{title}</h{num}>\n')
        
def clear():
    if gen_report:
        with open("results.html", "w") as f:
            f.write("")

def line():
    if gen_report:
        with open("results.html", "a") as f:
            f.write('<hr style="border:1px solid #444; margin:10px 0;">\n')
        
        
def write(text):
    html_text = text.replace("\n", "<br>\n")
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f"{html_text}\n")


def add_plot(path="plot.png"):
    """Embed an image file directly into results.html using base64."""
    with open(path, "rb") as img:
        encoded = base64.b64encode(img.read()).decode("utf-8")

    html_img = (
        '<img src="data:image/png;base64,' +
        encoded +
        '" style="max-width:650px; height:auto;"><br>\n'
    )

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_img)
        
    if os.path.exists(path):
        os.remove(path)

In [186]:
import os
import shutil

def remove_expr(res_df):
    for path in res_df["path"]:
        full_path = os.path.join("experiments", path)

        if not os.path.isdir(full_path):
            print(f"[SKIP] Not found: {full_path}")
            continue

        answer = input(f"⚠️ Delete expr '{path}' ? (y/n): ").strip().lower()

        if answer == "y":
            shutil.rmtree(full_path)
            print(f"✅ Deleted: {path}")
        else:
            print(f"❌ Skipped: {path}")

In [187]:
from itertools import product
import pandas as pd


def extract_maxes(df, cols=["seed", "agent_a_training_mode"], max_col="mutual_play_accuracy"):
    values = []
    for col in cols:
        value = set(df[col])
        values.append(value)

    combinations = [list(x) for x in product(*values)]

    maxes = []
    for comb in combinations:
        section = filter_df({col: value for col, value in zip(cols, comb)}, df)
        if not section.empty:
            max_row = section.loc[section[max_col].idxmax()]
            maxes.append(max_row.to_frame().T)

    final_df = pd.concat(maxes, ignore_index=True)
    return final_df.sort_values(by=cols)


def mean_and_std(df, out_cols=["VQEL", "dataset", "sim", "agent_a_training_mode"]):
    rows = []

    for i in range(0, len(df), 3):
        chunk = df.iloc[i:i+3]

        mean = chunk["mutual_play_accuracy"].mean() * 100
        std = chunk["mutual_play_accuracy"].std() * 100

        rows.append({
            col: chunk[col].iloc[0] for col in out_cols} | {
            "mutual_play_accuracy": f"{mean:.1f} ± {std:.1f}"
        })

    out = pd.DataFrame(rows)
    return out.sort_values(by=out_cols)
    

In [188]:
clear()

---

# EXP1:

In [ ]:
add_heading(1, "EXP1: ")

write(
"""
"""
)

In [190]:
vq_cols = [
    "seed",
    "VQEL",
    "dataset",
    "sim",
    "message_length",
    "agent_a_training_mode", 
    "learning_rate_phase1",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
    "num_iterations",
    "learning_rate_tt",
    "test_time_training_accuracy",
    "message_length_tt",
    "sampling_temperature",
    "representation_dim",
    "pretrained_checkpoint_a",
    "path",
]

vq_rf_cols = [
    "seed",
    "VQEL",
    "dataset",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "mutual_play_accuracy",
    "commitment_weight",
    "sampling_temperature",
    "representation_dim",
    "agent_a_training_mode",
    "num_pretrain_epochs",
    "contrastive_loss_temperature",
    "path"
]

bs_cols = [
    "VQEL",
    "dataset",
    "contrastive_loss_temperature",
    "entropy_regularization_factor",
    "learning_rate",
    "test_accuracy",
    "sampling_temperature",
    "representation_dim",
    "path"
]

vq_rf_cols_report = [
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "commitment_weight",
    "sampling_temperature",
    "mutual_play_accuracy",
]

vq_cols_report = [
    "agent_a_training_mode",
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
]

bs_cols_report = [
    "learning_rate",
    "test_accuracy",
]

## ShapeWorld

### VQEL - Euclidean

In [194]:
add_heading(2, "ShapeWorld")
add_heading(3, "VQEL - Euclidean")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "VQEL": True,
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,message_length,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,num_iterations,learning_rate_tt,test_time_training_accuracy,message_length_tt,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
21,1,True,shape,euclidean,"[1, 2, 3, 4]",frozen,1e-03,1e-03,0.607,0.726,0,-,0.765,5,1e-05,1024,None,20251221_0002_bs32_vocab10_repr1024_lr1_0.001_...
9,1,True,shape,euclidean,"[1, 2, 3, 4]",frozen,1e-04,1e-04,0.284,0.542,0,-,0.541,5,1e-05,1024,None,20251221_0016_bs32_vocab10_repr1024_lr1_0.0001...
24,1,True,shape,euclidean,"[1, 2, 3, 4]",frozen,1e-05,1e-05,0.323,0.579,0,-,0.607,5,1e-05,1024,None,20251221_0029_bs32_vocab10_repr1024_lr1_1e-05_...
5,1,True,shape,euclidean,"[2, 3, 4]",frozen,1e-03,1e-03,0.664,0.789,0,-,0.746,5,1e-05,1024,None,20251221_0043_bs32_vocab10_repr1024_lr1_0.001_...
14,1,True,shape,euclidean,"[2, 3, 4]",frozen,1e-04,1e-04,0.349,0.679,0,-,0.682,5,1e-05,1024,None,20251221_0058_bs32_vocab10_repr1024_lr1_0.0001...
0,1,True,shape,euclidean,"[2, 3, 4]",frozen,1e-05,1e-05,0.339,0.613,0,-,0.639,5,1e-05,1024,None,20251221_0113_bs32_vocab10_repr1024_lr1_1e-05_...
23,1,True,shape,euclidean,"[3, 4]",frozen,1e-03,1e-03,0.752,0.798,0,-,0.82,5,1e-05,1024,None,20251221_0128_bs32_vocab10_repr1024_lr1_0.001_...
27,1,True,shape,euclidean,"[3, 4]",frozen,1e-04,1e-04,0.622,0.725,0,-,0.688,5,1e-05,1024,None,20251221_0144_bs32_vocab10_repr1024_lr1_0.0001...
7,1,True,shape,euclidean,"[3, 4]",frozen,1e-05,1e-05,0.392,0.657,0,-,0.681,5,1e-05,1024,None,20251221_0200_bs32_vocab10_repr1024_lr1_1e-05_...
16,1,True,shape,euclidean,[4],frozen,1e-03,1e-03,0.804,0.847,0,-,0.74,5,1e-05,1024,None,20251221_0216_bs32_vocab10_repr1024_lr1_0.001_...


In [195]:
final = extract_maxes(res, cols=["agent_a_training_mode", "message_length"])
final[vq_cols]

,seed,VQEL,dataset,sim,message_length,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,num_iterations,learning_rate_tt,test_time_training_accuracy,message_length_tt,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
3,1,True,shape,euclidean,"[1, 2, 3, 4]",frozen,1e-03,1e-03,0.607,0.726,0,-,0.765,5,1e-05,1024,None,20251221_0002_bs32_vocab10_repr1024_lr1_0.001_...
2,1,True,shape,euclidean,"[2, 3, 4]",frozen,1e-03,1e-03,0.664,0.789,0,-,0.746,5,1e-05,1024,None,20251221_0043_bs32_vocab10_repr1024_lr1_0.001_...
0,1,True,shape,euclidean,"[3, 4]",frozen,1e-03,1e-03,0.752,0.798,0,-,0.82,5,1e-05,1024,None,20251221_0128_bs32_vocab10_repr1024_lr1_0.001_...
1,1,True,shape,euclidean,[4],frozen,1e-03,1e-03,0.804,0.847,0,-,0.74,5,1e-05,1024,None,20251221_0216_bs32_vocab10_repr1024_lr1_0.001_...


### VQEL - Cosine

In [196]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,message_length,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,num_iterations,learning_rate_tt,test_time_training_accuracy,message_length_tt,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
26,1,True,shape,cosine,"[1, 2, 3, 4]",frozen,1e-03,1e-03,0.754,0.817,0,-,0.828,5,1e-05,1024,None,20251220_1908_bs32_vocab10_repr1024_lr1_0.001_...
19,1,True,shape,cosine,"[1, 2, 3, 4]",frozen,1e-03,1e-04,0.754,0.832,0,-,0.837,5,1e-05,1024,None,20251220_1840_bs32_vocab10_repr1024_lr1_0.001_...
1,1,True,shape,cosine,"[1, 2, 3, 4]",frozen,1e-03,1e-05,0.754,0.806,0,-,0.818,5,1e-05,1024,None,20251220_1922_bs32_vocab10_repr1024_lr1_0.001_...
8,1,True,shape,cosine,"[1, 2, 3, 4]",frozen,1e-04,1e-04,0.684,0.838,0,-,0.853,5,1e-05,1024,None,20251220_1854_bs32_vocab10_repr1024_lr1_0.0001...
18,1,True,shape,cosine,"[2, 3, 4]",frozen,1e-03,1e-03,0.81,0.872,0,-,0.853,5,1e-05,1024,None,20251220_0805_bs32_vocab10_repr1024_lr1_0.001_...
13,1,True,shape,cosine,"[2, 3, 4]",frozen,1e-03,1e-04,0.81,0.9,0,-,0.876,5,1e-05,1024,None,20251220_0704_bs32_vocab10_repr1024_lr1_0.001_...
12,1,True,shape,cosine,"[2, 3, 4]",frozen,1e-04,1e-04,0.733,0.86,0,-,0.891,5,1e-05,1024,None,20251220_0820_bs32_vocab10_repr1024_lr1_0.0001...
17,1,True,shape,cosine,"[3, 4]",frozen,1e-03,1e-03,0.841,0.865,0,-,0.871,5,1e-05,1024,None,20251220_0631_bs32_vocab10_repr1024_lr1_0.001_...
22,1,True,shape,cosine,"[3, 4]",frozen,1e-03,1e-04,0.841,0.889,0,-,0.891,5,1e-05,1024,None,20251220_0615_bs32_vocab10_repr1024_lr1_0.001_...
20,1,True,shape,cosine,"[3, 4]",frozen,1e-04,1e-04,0.743,0.871,0,-,0.879,5,1e-05,1024,None,20251220_0648_bs32_vocab10_repr1024_lr1_0.0001...


In [197]:
final = extract_maxes(res, cols=["agent_a_training_mode", "message_length"])
final[vq_cols]

,seed,VQEL,dataset,sim,message_length,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,num_iterations,learning_rate_tt,test_time_training_accuracy,message_length_tt,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
3,1,True,shape,cosine,"[1, 2, 3, 4]",frozen,1e-04,1e-04,0.684,0.838,0,-,0.853,5,1e-05,1024,None,20251220_1854_bs32_vocab10_repr1024_lr1_0.0001...
2,1,True,shape,cosine,"[2, 3, 4]",frozen,1e-03,1e-04,0.81,0.9,0,-,0.876,5,1e-05,1024,None,20251220_0704_bs32_vocab10_repr1024_lr1_0.001_...
0,1,True,shape,cosine,"[3, 4]",frozen,1e-03,1e-04,0.841,0.889,0,-,0.891,5,1e-05,1024,None,20251220_0615_bs32_vocab10_repr1024_lr1_0.001_...
1,1,True,shape,cosine,[4],frozen,1e-03,1e-04,0.857,0.907,0,-,0.862,5,1e-05,1024,None,20251220_1028_bs32_vocab10_repr1024_lr1_0.001_...
